## Smart Restaurant Finder

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ollama
import warnings; warnings.filterwarnings("ignore")

In [2]:
# This downloads the model if you don't have it yet
ollama.pull('qwen2.5:1.5b')

ProgressResponse(status='success', completed=None, total=None, digest=None)

In [3]:
df_reviews = pd.read_json(
    "yelp_academic_dataset_review1.json", 
    lines=True, 
    nrows=50000
)

df_business = pd.read_json(
    "yelp_academic_dataset_business1.json", 
    lines=True
)

In [6]:
df_restaurants = pd.merge(df_reviews, df_business , on='business_id', how='inner', suffixes=('_review', '_business'))


In [7]:
df_restaurants.isnull().sum()

review_id            0
user_id              0
business_id          0
stars_review         0
useful               0
funny                0
cool                 0
text                 0
date                 0
name                 0
address              0
city                 0
state                0
postal_code          0
latitude             0
longitude            0
stars_business       0
review_count         0
is_open              0
attributes         768
categories           1
hours             2848
dtype: int64

### Preparing the Data


In [19]:

# 1.10 reviews per restaurant

top_reviews = df_restaurants.groupby('business_id')['text'].agg(list).reset_index()
top_reviews['text'] = top_reviews['text'].str[:10]
top_reviews['combined_reviews'] = top_reviews['text'].str.join(' | ')
top_reviews = top_reviews.drop(columns='text')

# 2. Restaurant Info
business_info = df_restaurants.drop_duplicates(subset='business_id')[[
    'business_id', 'name', 'city', 'state', 'stars_business', 'categories', 'address'
]]

df_restclean = pd.merge(business_info, top_reviews, on='business_id')


df_restclean['Description'] = (
    'Restaurant: ' + df_restclean['name'] + '; ' +
    'Location: ' + df_restclean['city'] + ', ' + df_restclean['state'] + '; ' +
    'Rating: ' + df_restclean['stars_business'].astype(str) + ' stars; ' +
    'Categories: ' + df_restclean['categories'] + '; ' +
    'Reviews: ' + df_restclean['combined_reviews']
)



In [9]:
# Limpiar NaN de la columna Description
df_restclean = df_restclean.dropna(subset=['Description'])

# Asegurarse que sea string
df_restclean['Description'] = df_restclean['Description'].astype(str)

In [20]:
df_restclean.head(1)

,business_id,name,city,state,stars_business,categories,address,combined_reviews,Description
0,XQfwVwDr-v0ZS3_CbbE5Xw,Turning Point of North Wales,North Wales,PA,3.0,"Restaurants, Breakfast & Brunch, Food, Juice B...",1460 Bethlehem Pike,"If you decide to eat here, just be aware it is...",Restaurant: Turning Point of North Wales; Loca...


### Embeddings


In [10]:
from sentence_transformers import SentenceTransformer

# Get our descriptions as a list
df_rest_description = df_restclean['Description'].to_list()

# Load a pre-trained embedding model (MiniLM by Microsoft, free to use)
# This model is specialized for creating embeddings, it's not a chatbot
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

print("Generating embeddings for all restaurants...")
embeddings = embedding_model.encode(df_rest_description, show_progress_bar=True)

print(f"Created {len(embeddings)} embeddings, each with {embeddings[0].shape[0]} numbers")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generating embeddings for all restaurants...


Batches:   0%|          | 0/251 [00:00<?, ?it/s]

Created 8025 embeddings, each with 384 numbers


### Creating the database


In [11]:
import chromadb

# Create our database
client = chromadb.Client()


collection = client.get_or_create_collection(name="restaurants_123")

In [12]:
# Prepare the data for ChromaDB

print(df_restclean.shape)

ids        = df_restclean['business_id'].tolist()
documents  = df_restclean['Description'].tolist()
metadatas  = df_restclean[['stars_business', 'city', 'categories']].to_dict(orient='records')
emb_list   = [emb.tolist() for emb in embeddings]

# Add in chunks (ChromaDB has a limit of ~5,000 records at a time)
for start in range(0, len(ids), 5000):
    end = start + 5000
    collection.add(
        ids        = ids[start:end],
        documents  = documents[start:end],
        metadatas  = metadatas[start:end],
        embeddings = emb_list[start:end],
    )
print(f"Loaded {collection.count()} restaurants into the database.")

(8025, 9)
Loaded 8025 restaurants into the database.


### Finding the Best Matches

When a user types a query, we:
1. Convert the query into an embedding (using the same MiniLM model)
2. Ask ChromaDB to find the top-k closest movie embeddings
3. Return the matching movies


In [13]:
def retrieve_relevant_records(query, k=5, filter_metadata=None):
    # Convert the query into an embedding
    query_embedding = embedding_model.encode([query])
    
    # Build the filter
    where_document = {}
    if filter_metadata:
        where_document = filter_metadata
    
    # Search ChromaDB for the closest matches
    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=k,
        where=where_document if where_document else None
    )
    
    descriptions = results["documents"][0]
    ids          = results["ids"][0]
    rows         = df_restclean[df_restclean['business_id'].isin(ids)]
    
    return descriptions, rows

In [14]:
query = "Suggest a well-rated restaurant with good Italian food."
descriptions, df_selected = retrieve_relevant_records(
    query,
    k=5,
    filter_metadata={"stars_business": {"$gte": 4.0}}
)
df_selected

,business_id,name,city,state,stars_business,categories,address,combined_reviews,Description
454,NcO-pWiZmNM5zBg8H9zB6Q,La Tavola Ristorante Italiano,Smyrna,TN,4.0,"Food, Pizza, Event Planning & Services, Italia...",114 Front St,We went on a Friday night and the place was fu...,Restaurant: La Tavola Ristorante Italiano; Loc...
1046,uKAbrDSJJzZliY1Yqu5KxQ,Scannicchio's,Philadelphia,PA,4.0,"Seafood, Italian, Restaurants, Mediterranean",2500 S Broad & Porter,Not mediocre at all IMHO. Makes me wonder if ...,Restaurant: Scannicchio's; Location: Philadelp...
2132,GJnYFvk9kbM182bKMFSAvg,Peno,Clayton,MO,4.5,"Italian, Soul Food, Sardinian, Calabrian, Rest...",7600 Wydown Blvd,We tried this place out because it was in the ...,"Restaurant: Peno; Location: Clayton, MO; Ratin..."
5210,nB9dev-4Wzxo4NHJkQQmUQ,Market on Ninth,Philadelphia,PA,4.5,"American (New), Coffee & Tea, Specialty Food, ...",943 S 9th St,LOVE this new Italian Market restaurant. The w...,Restaurant: Market on Ninth; Location: Philade...
5251,ONmS8r1C0F0vglz2nKjkPA,Rotolo's Pizzeria,Belle Chasse,LA,4.0,"Pizza, Restaurants","102 Woodland Hwy, Ste 5",The customer service is outstanding and the fo...,Restaurant: Rotolo's Pizzeria; Location: Belle...


In [33]:

def generate_answer(query, context):
    prompt = f"""
You are a restaurant recommendation assistant to help find a great place to eat.
Based ONLY on the following restaurant descriptions, answer the question.
As long as the question refers to restaurants more generally, and RESTAURANT DESCRIPTIONS is not "empty", mention one of 
the restaurants in the list as a recommendation.
Write in a very exciting and casual way and add two sentences explaining why the recommended restaurant is relevant to the query.
You are allowed to say "No match found"
If the user asks something unrelated to restaurants or food, say "No match found".
ALWAYS mention the exact city and state of the restaurant as it appears in the descriptions.
If the restaurant is not in the city the user asked for, say so clearly.

RESTAURANT DESCRIPTIONS:
{context}

QUESTION: {query}
"""
    response = ollama.generate(
        model="qwen2.5:1.5b",
        prompt=prompt,
        options={'temperature': 0.1}
    )
    return response['response']

In [22]:
def rag_query(query, k=5, filter_metadata=None):
    # Step 1: Retrieve the most relevant restaurants
    descriptions, rows = retrieve_relevant_records(query, k, filter_metadata)
    
    # Step 2: Build the context for the LLM
    if len(descriptions) == 0:
        context = "empty"
    else:
        context = "\n\n".join(descriptions)
    
    # Step 3: Generate the answer
    answer = generate_answer(query, context)
    return answer, rows

## *FINAL PRODUCT*

In [30]:
query = "Recommend me Argentinian restaurants in toronto"

answer, dfMatches = rag_query(
    query,
    k=5,
    filter_metadata={"stars_business": {"$gte": 3.5}}
)

print("Recommendation:\n", answer)
print("\nTop matches:")
dfMatches

Recommendation:
 No match found

Top matches:


,business_id,name,city,state,stars_business,categories,address,combined_reviews,Description
314,VVvUBlc_WIEb8obKGq39dA,Four Green Fields,Tampa,FL,4.5,"Irish Pub, Bars, Restaurants, Irish, Nightlife","4100 George J Bean Pkwy, Airside E, Tampa Inte...",Nice little bar located at the very far end of...,Restaurant: Four Green Fields; Location: Tampa...
1405,GgcvRnt5_z3NEC0D6vNncQ,Cafe Buenos Aires,Santa Barbara,CA,3.5,"Restaurants, Argentine",1316 State St,"A very sweet find on the main drag, State Stre...",Restaurant: Cafe Buenos Aires; Location: Santa...
4274,BVF9gWOAItS-781FlzGA_w,Tango Grill Nashville,Nashville,TN,5.0,"Argentine, Restaurants","4930 Linbar Dr, Ste 102",The quality and diversity of the menu for the ...,Restaurant: Tango Grill Nashville; Location: N...
4409,B73uHrYlkCNOIS6JwGBqYg,Barrio Cuisine,Tucson,AZ,3.5,"Tapas/Small Plates, American (Traditional), La...",188 E Broadway Blvd,Best Chelada I've ever tasted! atmosphere is i...,"Restaurant: Barrio Cuisine; Location: Tucson, ..."
6261,fM3rm3tcEcEOaH1mSrXkrA,Alma Llanera Venezuelan Food,Pinellas Park,FL,4.5,"Restaurants, Latin American, Venezuelan",6571 102nd Ave N,So happy we got some Venezuelan while in Tampa...,Restaurant: Alma Llanera Venezuelan Food; Loca...


### Building a Web Interface with Gradio

In [24]:
import gradio as gr

def gradio_rag(query, min_rating, k):
    filter_metadata = {
        "stars_business": {"$gte": float(min_rating)}
    }
    answer, rows = rag_query(
        query=query,
        k=int(k),
        filter_metadata=filter_metadata,
    )
    display_cols = ["name", "city", "state", "stars_business", "Description"]
    rows = rows.reset_index(drop=True)[display_cols]
    return answer.strip(), rows

with gr.Blocks(title="Restaurant Recommender", theme=gr.themes.Soft(
    primary_hue="orange",
    secondary_hue="red",
)) as demo:
    gr.Markdown(
        """
        ## 🍽️ Restaurant Recommender  
        Enter what you feel like eating, set the filters, and let AI suggest a restaurant.
        """
    )
    with gr.Row():
        query_box = gr.Textbox(
            label="Your query",
            placeholder="For example: I want a good Italian restaurant with great pasta",
            lines=2,
        )
    with gr.Row():
        min_rating_slider = gr.Slider(
            minimum=1.0,
            maximum=5.0,
            value=4.0,
            step=0.5,
            label="Minimum rating",
        )
        k_slider = gr.Slider(
            minimum=1,
            maximum=10,
            value=5,
            step=1,
            label="How many recommendations do you want?",
        )
    run_button = gr.Button("Find Restaurant 🍴")
    gr.Markdown("### What we found:")
    answer_box = gr.Markdown(label="Answer")
    table_output = gr.Dataframe(
        headers=["Name", "City", "State", "Rating", "Description"],
        datatype=["str", "str", "str", "float", "str"],
        label="Matching restaurants",
        interactive=False,
    )
    run_button.click(
        fn=gradio_rag,
        inputs=[query_box, min_rating_slider, k_slider],
        outputs=[answer_box, table_output],
    )

demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://463571689077c09f81.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
